# ASGC on CIFAR-10 / ResNet-18 - frozen results

**This notebook is a results viewer, not a training run.** The CIFAR-10 experiments are compute-heavy and were already executed (single seed, seed 0); the numbers below are copied from [`findings_for_report.md`](../findings_for_report.md) sections 4 and 5. Nothing here downloads CIFAR-10 or trains a model.

**Provenance.** Research simulation `src/asgc/`; CIFAR-variant ResNet-18 (11.17 M parameters, 44.7 MB per uncompressed gradient); 3 worker processes; worker 1 delayed by +0.5 s and worker 2 by +1.5 s; 6000 async updates (~15 epochs); SGD lr 0.02; RTX 4050 laptop GPU. Runs were written to `results/cifar_*` and `results/*_bw250` (gitignored) with `events.jsonl` as the source of truth.

**Two regimes are reported.** The unconstrained table asks how much communication compression saves when the network is free. The 250 Mbps table charges every push `payload_bytes / bandwidth` and asks whether compression converts into wall-clock. Results differ qualitatively, so deployment claims must state the regime.


## 1. Unconstrained network (6000 updates)

`max eval drop` is the largest single evaluation-accuracy drop observed during the run. Compression ratio (CR) is exact encoded bytes relative to the uncompressed up-link.


In [ ]:
import pandas as pd

# Frozen from findings_for_report.md section 4 (single seed, seed 0).
free = [
    {"method": "Plain async",                 "accuracy": 85.50, "gb": 268.2, "cr": 1.00, "upd_s": 5.38, "max_eval_drop_pts": 0.4},
    {"method": "Fixed Top-K (no EF)",         "accuracy": 84.32, "gb": 134.1, "cr": 2.00, "upd_s": 5.16, "max_eval_drop_pts": 0.0},
    {"method": "Top-K + EF",                  "accuracy": 86.14, "gb": 134.1, "cr": 2.00, "upd_s": 5.00, "max_eval_drop_pts": 0.0},
    {"method": "Adaptive (rho floor 0.05)",   "accuracy": 83.72, "gb": 34.0,  "cr": 8.13, "upd_s": 4.95, "max_eval_drop_pts": 0.0},
]
free_df = pd.DataFrame(free)
free_df.style.format({
    "accuracy": "{:.2f}%", "gb": "{:.1f}", "cr": "{:.2f}x",
    "upd_s": "{:.2f}", "max_eval_drop_pts": "{:.1f}",
}).highlight_max(subset=["accuracy"], color="#d7f5d7").highlight_min(subset=["gb"], color="#d7f5d7")


### Findings from the unconstrained table

- **F5 - The central claim replicates on the main workload.** Top-K + EF beats uncompressed async on both axes: +0.64 accuracy points at exactly half the bytes, with zero stability drops across the run.
- **F6 - Error feedback's contribution is larger on the harder model.** Fixed Top-K without EF loses 1.18 points to uncompressed (85.50 -> 84.32); adding EF at identical bytes gains +1.82 points to 86.14%. That is roughly a 3-point swing on CIFAR versus about 0.5 on Fashion-MNIST.
- **F7 - The adaptive controller adapted cleanly at scale but chose the wrong operating point for the accuracy gate.** Zero false instability trips across 40 logged decisions; rho moved 0.25 -> 0.05 (its floor) and S_max 4 -> 10 within two minutes, then held stably. It found 8.13x compression - four times less data than the best fixed policy - at a cost of 2.4 accuracy points.
- **RQ3, unconstrained:** joint adaptation did *not* beat fixed Top-K + EF on wall-clock. The controller's overhead and conservative drift make it slower when the link is not the bottleneck.


## 2. 250 Mbps up-link constraint

Every push sleeps `payload_bytes / bandwidth`. `time_to_0.80` is the wall-clock to 80% test accuracy (the 93% aspiration is unreachable at this 15-epoch budget).


In [ ]:
# Frozen from findings_for_report.md section 5 (single seed, seed 0).
bw250 = [
    {"method": "Plain async",               "accuracy": 86.01, "wall_min": 70.4, "time_to_080_min": 41.1, "gb": 268.2, "cr": 1.00},
    {"method": "Fixed Top-K (no EF)",       "accuracy": 84.60, "wall_min": 54.0, "time_to_080_min": 24.2, "gb": 134.1, "cr": 2.00},
    {"method": "Top-K + EF",                "accuracy": 86.50, "wall_min": 48.8, "time_to_080_min": 20.5, "gb": 134.1, "cr": 2.00},
    {"method": "Adaptive (rho floor 0.05)", "accuracy": 83.76, "wall_min": 26.7, "time_to_080_min": 13.7, "gb": 29.8,  "cr": 9.01},
]
bw_df = pd.DataFrame(bw250)
bw_df["speedup_to_080"] = bw_df.loc[0, "time_to_080_min"] / bw_df["time_to_080_min"]
bw_df.style.format({
    "accuracy": "{:.2f}%", "wall_min": "{:.1f} min", "time_to_080_min": "{:.1f} min",
    "gb": "{:.1f}", "cr": "{:.2f}x", "speedup_to_080": "{:.2f}x",
}).highlight_max(subset=["accuracy"], color="#d7f5d7").highlight_min(subset=["time_to_080_min", "gb"], color="#d7f5d7")


### Findings from the bandwidth-constrained table

- **F8 - Under bandwidth constraints the project beats plain async on every axis, including wall-clock.** Top-K + EF gains +0.49 accuracy points, uses 31% less wall time and half the bytes, and reaches 80% accuracy 1.98x faster.
- **F9 - The adaptive method converts compression into wall-clock directly.** Its throughput rose during the run (1.4 -> 3.75 updates/s) as the controller compressed harder under pressure. It finished 2.6x faster than plain async at 9x compression and 3x faster to 80% accuracy.
- **F10 - Compression's wall-clock value exists only when the link is the bottleneck.** With no communication delay the total-time column is flat across methods; at 250 Mbps the same experiments separate by up to 2.6x. Always state the network regime.


## 3. Figures from the frozen tables

The left panel is the free-network trade-off (accuracy versus up-link bytes); the right panel is the bandwidth-constrained regime (accuracy versus wall-clock). Both are drawn from the embedded constants.


In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(13, 4.8))
free_colors = ["#4c78a8", "#8a8f98", "#2e7d32", "#f58518"]
bw_colors = ["#4c78a8", "#8a8f98", "#2e7d32", "#f58518"]
short = ["plain\nasync", "fixed\nTop-K", "Top-K\n+ EF", "adaptive"]

ax = axes[0]
ax.scatter(free_df["gb"], free_df["accuracy"], s=90, c=free_colors, zorder=3)
for _, row in free_df.iterrows():
    ax.annotate(row["method"], (row["gb"], row["accuracy"]), textcoords="offset points",
                xytext=(6, 6), fontsize=8)
ax.set_xlabel("up-link GB")
ax.set_ylabel("test accuracy (%)")
ax.set_title("Unconstrained: accuracy vs bytes")
ax.grid(alpha=0.3, linestyle="--")

ax = axes[1]
ax.bar(short, bw_df["time_to_080_min"], color=bw_colors)
for i, v in enumerate(bw_df["time_to_080_min"]):
    ax.text(i, v + 1.0, f"{v:.1f}", ha="center", fontsize=8)
ax.set_ylabel("minutes to 80% accuracy")
ax.set_title("250 Mbps: time-to-target")
ax.tick_params(axis="x", labelsize=8)

for ax in axes:
    ax.spines[["top", "right"]].set_visible(False)
fig.tight_layout()
plt.show()


## 4. Caveats specific to these runs

- **Single seed.** All CIFAR rows are one run; the multi-seed protocol is `configs/matrix_cifar_full.yaml`, which has not been run. Orderings between methods within 0.5 points are within noise.
- **Synchronous CIFAR baseline was not run.** The 10-16x asynchrony claim comes from Fashion-MNIST and the transformer slice; it is not measured here.
- **Accuracy ceiling.** 6000 updates is roughly 15 epochs, so the runs top out near 86%, well below the 93% aspiration. `time_to_0.80` is the meaningful target.
- **Adaptive operating point is untuned.** The rho floor 0.05 was chosen on Fashion-MNIST and lets CIFAR compress past the accuracy knee: 8.13x at -2.4 points (free network) and 9.01x at -2.7 points (250 Mbps). A floor near 0.10-0.15 is the likely fix.
- **BatchNorm recalibration.** The server never runs training forwards, so BN statistics are recalibrated with a 64-batch sweep before each evaluation. It is honest per method but an approximation of fully distributed BN.
- **Byte model.** Only the compressed up-link is charged; down-link snapshots are unmodeled and identical across methods. The harness has no acknowledgement, so rejected updates cannot roll back their residual (findings section 12, item 4).
- **Eval cadence.** CIFAR evaluates every 500 versions (about 95 s) while the controller runs every 30 s, so many control windows see no fresh evaluation (RQ5).


## 5. Reproduce from raw telemetry (optional)

These commands rerun the experiments and are **not** executed by this notebook. They require the repository, `pip install -r requirements.txt`, and ideally a GPU.

```bash
# Unconstrained baselines and adaptive method
python scripts/run_experiment.py --config configs/baselines/cifar_async_plain.yaml
python scripts/run_experiment.py --config configs/baselines/cifar_async_fixed_topk.yaml
python scripts/run_experiment.py --config configs/comparisons/cifar_slice/cifar_async_topk_ef.yaml
python scripts/run_experiment.py --config configs/adaptive/cifar_adaptive_topk.yaml

# 250 Mbps up-link variants
python scripts/run_experiment.py --config configs/comparisons/cifar_bandwidth/cifar_async_plain_bw250.yaml
python scripts/run_experiment.py --config configs/comparisons/cifar_bandwidth/cifar_async_fixed_topk_bw250.yaml
python scripts/run_experiment.py --config configs/comparisons/cifar_bandwidth/cifar_async_topk_ef_bw250.yaml
python scripts/run_experiment.py --config configs/comparisons/cifar_bandwidth/cifar_adaptive_topk_bw250.yaml

# Aggregate
python scripts/plot_results.py --runs cifar_async_plain cifar_async_fixed_topk cifar_async_topk_ef cifar_adaptive_topk --target 0.80
```

`run_experiment.py` writes `results/<run.name>/events.jsonl`; `plot_results.py` reads runs by name and writes `results/plot/summary.csv` plus comparison figures. The multi-seed matrix is `python scripts/run_matrix.py --matrix configs/matrix_cifar_full.yaml`.
